<a href="https://colab.research.google.com/github/Rini43/Insurance/blob/main/damage_detection_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

# Libraries

In [1]:
!pip install ultralytics opencv-python-headless pyyaml -q
import os, shutil, time
import yaml
import random
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


# Read Dataset

In [2]:
import os
import shutil
import time

# Google Drive project location
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO'

# Local Colab copy
LOCAL_PROJECT_ROOT = '/content/CARDD_YOLO'

# Check that Google Drive is mounted
assert os.path.isdir('/content/drive/MyDrive'), \
    'Google Drive is not mounted. Run: from google.colab import drive; drive.mount("/content/drive")'

# Check that the project exists in Drive
assert os.path.isdir(DRIVE_PROJECT_ROOT), \
    f'CARDD_YOLO project not found at:\n{DRIVE_PROJECT_ROOT}'

# Copy project from Drive to Colab local storage
if os.path.isdir(LOCAL_PROJECT_ROOT):
    print(f'Local copy already exists at {LOCAL_PROJECT_ROOT}')
    print('Skipping copy. Delete the local folder if you want to copy it again.')
else:
    print('Copying CARDD_YOLO from Google Drive to local Colab storage...')
    start = time.time()

    shutil.copytree(DRIVE_PROJECT_ROOT, LOCAL_PROJECT_ROOT)

    print(f'Copy complete in {time.time() - start:.1f} seconds')

# Use the local copy for YOLO training
PROJECT_ROOT = LOCAL_PROJECT_ROOT

# Dataset YAML
yaml_path = os.path.join(PROJECT_ROOT, 'data.yaml')

# Check data.yaml exists
assert os.path.isfile(yaml_path), \
    f'data.yaml not found at:\n{yaml_path}'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('data.yaml:', yaml_path)


AssertionError: Google Drive is not mounted. Run: from google.colab import drive; drive.mount("/content/drive")

In [ ]:
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO'
LOCAL_PROJECT_ROOT = '/content/CARDD_YOLO'

assert os.path.isdir(DRIVE_PROJECT_ROOT), f'Drive dataset not found at {DRIVE_PROJECT_ROOT} — update the path'

if os.path.isdir(LOCAL_PROJECT_ROOT):
    print('Local copy already exists at', LOCAL_PROJECT_ROOT, '— skipping copy. Delete it first if you want to re-copy.')
else:
    print('Copying dataset from Drive to local disk — this happens once per session...')
    start = time.time()
    shutil.copytree(DRIVE_PROJECT_ROOT, LOCAL_PROJECT_ROOT)
    print(f'Copy complete in {time.time() - start:.1f} seconds')

PROJECT_ROOT = LOCAL_PROJECT_ROOT
yaml_path = f'{PROJECT_ROOT}/data.yaml'

##Validate the Local Dataset

In [ ]:
print('YOLO dataset root (local):', PROJECT_ROOT)
print('data.yaml exists:', os.path.exists(yaml_path))

for split in ['train', 'val', 'test']:
    image_dir = f'{PROJECT_ROOT}/images/{split}'
    label_dir = f'{PROJECT_ROOT}/labels/{split}'
    if os.path.isdir(image_dir) and os.path.isdir(label_dir):
        print(f'{split}: images={len(os.listdir(image_dir))}, labels={len(os.listdir(label_dir))}')
    else:
        print(f'{split}: NOT FOUND (image_dir exists={os.path.isdir(image_dir)}, label_dir exists={os.path.isdir(label_dir)})')

In [ ]:
assert os.path.exists(yaml_path), 'data.yaml not found in local copy'

with open(yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)

# data.yaml paths must point to LOCAL paths now, not Drive — rewrite and save a local version
data_yaml['train'] = f'{PROJECT_ROOT}/images/train'
data_yaml['val'] = f'{PROJECT_ROOT}/images/val'
if os.path.isdir(f'{PROJECT_ROOT}/images/test'):
    data_yaml['test'] = f'{PROJECT_ROOT}/images/test'

with open(yaml_path, 'w') as f:
    yaml.safe_dump(data_yaml, f)

print('data.yaml contents (paths rewritten to local disk):')
print(data_yaml)

assert os.path.isdir(f'{PROJECT_ROOT}/images/train'), 'Training images folder not found'
assert os.path.isdir(f'{PROJECT_ROOT}/labels/train'), 'Training labels folder not found'
print('\nLocal dataset is ready for training.')

##Detecting Label Format

In [ ]:
label_dir = f'{PROJECT_ROOT}/labels/train'
label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
sample_files = random.sample(label_files, min(20, len(label_files)))

line_lengths = []
for fname in sample_files:
    with open(os.path.join(label_dir, fname)) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                line_lengths.append(len(parts))

if not line_lengths:
    raise ValueError('No label content found — check that label .txt files are not empty')

max_len = max(line_lengths)
is_segmentation = max_len > 5

print(f'Sampled {len(sample_files)} label files, {len(line_lengths)} annotation lines')
print(f'Max fields per line: {max_len}')
print(f'Detected format: {"SEGMENTATION (polygon masks)" if is_segmentation else "DETECTION (bounding boxes)"}')

MODEL_NAME = 'yolov8n-seg.pt' if is_segmentation else 'yolov8n.pt'
print(f'\nModel to use: {MODEL_NAME}')

##Image Checking

In [ ]:
img_dir = f'{PROJECT_ROOT}/images/train'
img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
sample_img_name = random.choice(img_files)
sample_label_name = os.path.splitext(sample_img_name)[0] + '.txt'

img = cv2.imread(os.path.join(img_dir, sample_img_name))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]

label_path = os.path.join(label_dir, sample_label_name)
if os.path.exists(label_path):
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls = int(parts[0])
            coords = list(map(float, parts[1:]))
            if is_segmentation:
                pts = [(int(coords[i]*w), int(coords[i+1]*h)) for i in range(0, len(coords), 2)]
                for i in range(len(pts)):
                    cv2.line(img, pts[i], pts[(i+1) % len(pts)], (255, 0, 0), 2)
            else:
                xc, yc, bw, bh = coords
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)

plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.title(sample_img_name)
plt.axis('off')
plt.show()

#Model Training

In [ ]:
model = YOLO(MODEL_NAME)

results = model.train(
    data = yaml_path,
    epochs = 30,
    imgsz = 640,
    batch = 16,
    device = 0,
    project = '/content/runs',
    name = 'car_damage_yolov8',
    patience = 10
)

In [ ]:
## Resume training if a session disconnects mid-run
# last_ckpt = '/content/runs/car_damage_yolov8/weights/last.pt'
# model = YOLO(last_ckpt)
# results = model.train(resume=True)

#Evaluation

In [ ]:
best_model_path_local = '/content/runs/car_damage_yolov8/weights/best.pt'
trained_model = YOLO(best_model_path_local)

metrics = trained_model.val()

print('Box mAP50-95:', metrics.box.map)
print('Box mAP50:', metrics.box.map50)
if hasattr(metrics, 'seg') and metrics.seg is not None:
    print('Seg mAP50-95:', metrics.seg.map)
    print('Seg mAP50:', metrics.seg.map50)

print('\nPer-class results:')
names_iter = data_yaml.get('names', {}).items() if isinstance(data_yaml.get('names'), dict) else enumerate(data_yaml.get('names', []))
for i, name in names_iter:
    try:
        print(f'  {name}: mAP50-95 = {metrics.box.maps[i]:.3f}')
    except Exception:
        pass

In [ ]:
# Copy final weights and best model to Google drive
drive_runs_dest = f'{DRIVE_PROJECT_ROOT}/runs/car_damage_yolov8'

os.makedirs(os.path.dirname(drive_runs_dest), exist_ok=True)
if os.path.isdir(drive_runs_dest):
    shutil.rmtree(drive_runs_dest)

print('Copying trained model + logs to Drive (one-time write)...')
shutil.copytree('/content/runs/car_damage_yolov8', drive_runs_dest)

best_model_path = f'{drive_runs_dest}/weights/best.pt'
print('Saved to Drive at:', best_model_path)

#Testing

In [ ]:
# Testing on sample images
test_img_dir = f'{PROJECT_ROOT}/images/test'
if os.path.isdir(test_img_dir) and len(os.listdir(test_img_dir)) > 0:
    test_img_name = random.choice(os.listdir(test_img_dir))
    test_img_path = os.path.join(test_img_dir, test_img_name)
else:
    val_img_dir = f'{PROJECT_ROOT}/images/val'
    test_img_name = random.choice(os.listdir(val_img_dir))
    test_img_path = os.path.join(val_img_dir, test_img_name)

print('Running inference on:', test_img_path)
results = trained_model(test_img_path)
results[0].show()

In [ ]:
# Final Model path
print('Trained model permanently saved at (Drive):')
print(best_model_path)

# Description Training

In [ ]:
import os

ROOT = '/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO'

for root, dirs, files in os.walk(ROOT):
    level = root.replace(ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:10]:
        print(f"{indent}  {file}")

    if level > 3:
        dirs[:] = []


In [ ]:
from glob import glob
import os

ROOT = '/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO'

for pattern in [
    '**/*.yaml',
    '**/*.yml',
    '**/*.txt',
    '**/*.json',
    '**/*.csv'
]:
    files = glob(os.path.join(ROOT, pattern), recursive=True)

    print('\n', pattern)
    for f in files[:20]:
        print(f)


In [ ]:
print(open(
    '/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO/data.yaml'
).read())


In [ ]:
from glob import glob

labels = glob(
    '/content/drive/MyDrive/AI ML/Projects/CARDD_YOLO/**/*.txt',
    recursive=True
)

print(labels[:5])

if labels:
    print(open(labels[0]).read())
